# Fusion signature deconvolution on TCGA bulk RNA-seq

Applies the fused/parental/resistant scRNA-seq signature (`fusion_signature_expression.h5ad`) to real
TCGA bulk RNA-seq samples via two independent deconvolution methods -- NNLS against a fixed signature
matrix, and a Random Forest regressor trained directly on simulated pseudo-bulk mixtures (same approach
as `Fusion_signature_pseudobulk_deconvolution.ipynb`, Section 8) -- then compares estimated fused-cell
fraction across organ/tissue types and checks how well the two methods agree with each other.

Gene expression is streamed from [UCSC Xena](https://xenabrowser.net) with `xenaPython`, querying only
the signature gene panel (not the full ~50k-gene x ~10.5k-sample matrix), so nothing beyond a small
samples x signature-genes table is ever held in memory. `xenaPython` is not installed on the HPC, so an
offline/full-download fallback (plain `requests` + chunked `pandas`) is included at the bottom, commented
out, for running there instead.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import nnls
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

import xenaPython as xena

## 1. Load the fusion signature reference

Same signature-gene reference used in `Fusion_signature_pseudobulk_deconvolution.ipynb`. Unlike that
notebook, there's no held-out validation split here — TCGA's true fused fraction is exactly what we're
trying to estimate, so we use *all* reference cells to build the best possible per-class signature
matrix ("production" reference rather than a validation split).

In [ ]:
SIGNATURE_EXPRESSION_PATH = 'fusion_signature_expression.h5ad'

sig_adata = ad.read_h5ad(SIGNATURE_EXPRESSION_PATH)
signature_genes = sig_adata.var_names.tolist()
classes = sorted(sig_adata.obs['sample'].unique())
print(f'{len(signature_genes)} signature genes, {sig_adata.n_obs} reference cells')
print('Classes:', classes)

def to_linear(X):
    return np.sinh(X)

expr_asinh = sig_adata.X
if hasattr(expr_asinh, 'toarray'):
    expr_asinh = expr_asinh.toarray()
expr_linear = to_linear(expr_asinh)

sample_labels = sig_adata.obs['sample'].values
reference_matrix = np.column_stack([
    expr_linear[sample_labels == c].mean(axis=0)
    for c in classes
])  # (n_genes, n_classes)
reference_pool_mean = expr_linear.mean(axis=0)  # (n_genes,), all classes pooled - used for cross-platform rescaling below

print('Reference matrix shape:', reference_matrix.shape)

## 2. Pull TCGA sample/tissue metadata from Xena

`tcga_RSEM_gene_tpm` (hub `toilHub`) is the TCGA-only RSEM TPM matrix (log2(TPM+0.001)), 10,535 samples.
Its companion phenotype dataset `TcgaTargetGTEX_phenotype.txt` carries `_primary_site` (organ/tissue),
`_sample_type` (Primary Tumor / Solid Tissue Normal / Metastatic / ...), and `_study`.

Phenotype fields are categorical and come back from the hub as integer codes, decoded via `field_codes`.

In [ ]:
XENA_HOST = xena.PUBLIC_HUBS['toilHub']
EXPR_DATASET = 'tcga_RSEM_gene_tpm'
PHENO_DATASET = 'TcgaTargetGTEX_phenotype.txt'
PHENO_FIELDS = ['_primary_site', '_sample_type', '_study']
SAMPLE_TYPES_KEPT = ['Primary Tumor']  # drop Solid Tissue Normal / Metastatic / etc.
BATCH_SIZE = 1000

def decode_categorical(host, dataset, samples, fields, batch_size=BATCH_SIZE):
    code_lookup = {
        d['name']: d['code'].split('\t')
        for d in xena.field_codes(host, dataset, fields)
    }
    rows = []
    for i in range(0, len(samples), batch_size):
        batch = samples[i:i + batch_size]
        raw = xena.dataset_fetch(host, dataset, batch, fields)  # raw[f_idx][sample_idx] = code
        decoded = {
            field: [code_lookup[field][v] if v >= 0 else None for v in raw[f_idx]]
            for f_idx, field in enumerate(fields)
        }
        rows.append(pd.DataFrame({'sampleID': batch, **decoded}))
    return pd.concat(rows, ignore_index=True)

tcga_samples = xena.dataset_samples(XENA_HOST, EXPR_DATASET, None)
print(f'{len(tcga_samples)} total TCGA samples in {EXPR_DATASET}')

pheno_df = decode_categorical(XENA_HOST, PHENO_DATASET, tcga_samples, PHENO_FIELDS)
pheno_df = pheno_df[pheno_df['_study'] == 'TCGA']
pheno_df = pheno_df[pheno_df['_sample_type'].isin(SAMPLE_TYPES_KEPT)].reset_index(drop=True)

print(f'{len(pheno_df)} samples after filtering to {SAMPLE_TYPES_KEPT}')
print(pheno_df['_primary_site'].value_counts())

## 3. Stream signature-gene expression for the selected samples

Only `signature_genes` are requested per batch via `dataset_gene_probe_avg` — never the full matrix.

In [ ]:
bulk_sample_ids = pheno_df['sampleID'].tolist()

matched_genes = [g for g in signature_genes if g in xena.dataset_field(XENA_HOST, EXPR_DATASET)]
missing_genes = sorted(set(signature_genes) - set(matched_genes))
print(f'{len(matched_genes)}/{len(signature_genes)} signature genes found in {EXPR_DATASET}')
if missing_genes:
    print('Missing (dropped):', missing_genes)

expr_chunks = []
for i in range(0, len(bulk_sample_ids), BATCH_SIZE):
    batch = bulk_sample_ids[i:i + BATCH_SIZE]
    gene_records = xena.dataset_gene_probe_avg(XENA_HOST, EXPR_DATASET, batch, matched_genes)
    chunk = pd.DataFrame(
        {rec['gene']: rec['scores'][0] for rec in gene_records},
        index=batch,
    )
    expr_chunks.append(chunk)
    print(f'  fetched {min(i + BATCH_SIZE, len(bulk_sample_ids))}/{len(bulk_sample_ids)} samples', end='\r')

tcga_log2tpm = pd.concat(expr_chunks)[matched_genes]  # (n_samples, n_matched_genes), log2(TPM + 0.001)
tcga_linear = 2 ** tcga_log2tpm - 0.001
print(f'\nTCGA expression matrix: {tcga_linear.shape}')

## 3b. Match Gene Panel Positions

Both the NNLS reference matrix and the RF regressor's training features need to be restricted to
`matched_genes` (whatever subset of the signature was actually found in TCGA), in that exact order --
computed once here and reused by both.

In [ ]:
gene_idx = [signature_genes.index(g) for g in matched_genes]
print(f'gene_idx computed for {len(gene_idx)} matched genes (shared by NNLS and the RF regressor below)')

## 3c. Train a Random Forest Regressor for Direct Bulk-to-Composition Deconvolution

Same approach as `Fusion_signature_pseudobulk_deconvolution.ipynb` (Section 8): instead of solving a
signature-matrix regression (NNLS), fit a model directly on simulated pseudo-bulks built from these
same reference cells: `bulk expression -> composition`. Raw multi-output regression isn't constrained
to be non-negative or to sum to 1, so the model is trained on `log(true_fraction + eps)` and its raw
prediction is passed back through a softmax at inference time -- since `softmax(log(p)) = p`, this is
the correct inverse transform for compositional (simplex-valued) targets, not an ad hoc clamp.
`RandomForestRegressor` supports multi-output `y` natively, so no `MultiOutputRegressor` wrapper is
needed (unlike single-output regressors such as ElasticNet).

Pseudo-bulks are simulated from *all* reference cells (matching this notebook's "no held-out split,
build the best possible production reference" choice in Section 1), but restricted to `matched_genes`
so the trained regressor's input features line up exactly with what TCGA can actually provide -- feeding
it a differently-sized or differently-ordered feature vector later would silently misalign, not error.

Evaluated on a held-out split of the pseudo-bulks themselves (not TCGA samples, which have no ground
truth) -- report this held-out performance so the TCGA predictions below come with a sense of how
reliable this particular estimator actually is on data it wasn't fit on.

In [ ]:
RANDOM_SEED = 42
N_PSEUDOBULKS = 200
N_CELLS_PER_BULK = 200
DIRICHLET_ALPHA = 1.0            # uniform-over-simplex sampling of mixing fractions across pseudobulks
REGRESSION_TEST_FRACTION = 0.3   # held-out pseudo-bulks for evaluating the RF regressor
LOG_FRACTION_EPS = 1e-6          # floor added before log-transforming fractions for regression targets

rng = np.random.default_rng(RANDOM_SEED)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

all_cell_idx_by_class = {c: np.where(sample_labels == c)[0] for c in classes}
expr_linear_matched = expr_linear[:, gene_idx]  # (n_ref_cells, n_matched_genes)

pseudobulk_records = []
for b in range(N_PSEUDOBULKS):
    true_props = rng.dirichlet(np.full(len(classes), DIRICHLET_ALPHA))
    counts = rng.multinomial(N_CELLS_PER_BULK, true_props)
    true_fractions = counts / N_CELLS_PER_BULK

    drawn_indices = []
    for c, n_c in zip(classes, counts):
        if n_c == 0:
            continue
        drawn_indices.append(rng.choice(all_cell_idx_by_class[c], size=n_c, replace=True))
    drawn_indices = np.concatenate(drawn_indices) if drawn_indices else np.array([], dtype=int)

    bulk_profile_linear = expr_linear_matched[drawn_indices].mean(axis=0)
    pseudobulk_records.append({'true_fractions': true_fractions, 'bulk_profile_linear': bulk_profile_linear})

print(f'Simulated {len(pseudobulk_records)} pseudo-bulks of {N_CELLS_PER_BULK} cells each '
      f'(sampled from all {sig_adata.n_obs} reference cells, {len(gene_idx)} matched genes)')

true_frac_matrix = np.array([rec['true_fractions'] for rec in pseudobulk_records])
bulk_profile_matrix = np.array([rec['bulk_profile_linear'] for rec in pseudobulk_records])

X_reg = np.arcsinh(bulk_profile_matrix)
y_reg = np.log(true_frac_matrix + LOG_FRACTION_EPS)

train_idx_reg, test_idx_reg = train_test_split(
    np.arange(len(pseudobulk_records)), test_size=REGRESSION_TEST_FRACTION, random_state=RANDOM_SEED,
)

rf_reg = RandomForestRegressor(n_estimators=300, random_state=RANDOM_SEED)
rf_reg.fit(X_reg[train_idx_reg], y_reg[train_idx_reg])

test_pred_fractions = softmax(rf_reg.predict(X_reg[test_idx_reg]))
test_true_fractions = true_frac_matrix[test_idx_reg]

print(f'\nTrained RandomForestRegressor on {len(train_idx_reg)} pseudo-bulks, '
      f'evaluating on {len(test_idx_reg)} held out:')
for i, c in enumerate(classes):
    r, _ = pearsonr(test_true_fractions[:, i], test_pred_fractions[:, i])
    mae = np.abs(test_true_fractions[:, i] - test_pred_fractions[:, i]).mean()
    print(f'  {c}: held-out MAE={mae:.3f}, Pearson r={r:.3f}')

## 4. NNLS deconvolution per TCGA sample

Same NNLS approach as the pseudo-bulk notebook: solve `reference_matrix @ fractions ≈ bulk_profile` for
non-negative `fractions`, then normalize to sum to 1.

**Cross-platform caveat:** the reference is built from 10x scRNA-seq (asinh-transformed, per-cell
normalized) and the bulk data is RSEM TPM from a different assay entirely — the two are not on
inherently comparable absolute scales. Any single *per-sample* multiplicative scale difference cancels
out automatically once fractions are re-normalized to sum to 1, but *per-gene* scale differences between
platforms do not. As a lightweight correction, each gene's bulk values are rescaled by the ratio of its
mean in the scRNA reference pool to its mean across the bulk cohort, i.e. a simple mean-matching batch
correction — not a substitute for formal cross-platform normalization (e.g. quantile normalization to a
shared reference, as in CIBERSORTx), but enough to remove the biggest gene-level scale offsets. Treat
resulting fractions as indicative/relative estimates, not calibrated absolute percentages, unless this is
validated further (e.g. against pseudo-bulks built directly from bulk-scale data).

In [ ]:
bulk_pool_mean = tcga_linear.mean(axis=0).values  # (n_matched_genes,)
scale_factor = reference_pool_mean[gene_idx] / np.where(bulk_pool_mean == 0, np.nan, bulk_pool_mean)
scale_factor = np.nan_to_num(scale_factor, nan=1.0)

reference_matrix_matched = reference_matrix[gene_idx, :]

nnls_records = []
for sample_id, row in tcga_linear.iterrows():
    bulk_profile_rescaled = row.values * scale_factor
    coeffs, _residual = nnls(reference_matrix_matched, bulk_profile_rescaled)
    total = coeffs.sum()
    fractions = coeffs / total if total > 0 else np.zeros_like(coeffs)
    nnls_records.append({'sampleID': sample_id, **dict(zip(classes, fractions))})

nnls_df = pd.DataFrame(nnls_records)
tcga_results_df = pheno_df.merge(nnls_df, on='sampleID')
tcga_results_df = tcga_results_df.rename(columns={c: f'{c}_fraction' for c in classes})
tcga_results_df.to_csv('tcga_fusion_deconvolution_results.csv', index=False)
tcga_results_df.head()

## 4b. Apply the RF Regressor to TCGA Samples

Same `bulk_profile_rescaled` cross-platform correction as NNLS (reusing the already-computed
`scale_factor`), then `arcsinh`-transformed and fed through `rf_reg`, softmax'd back to fractions.
Unlike NNLS (which solves one system per sample), `RandomForestRegressor.predict` is vectorized --
all TCGA samples in a single call. Adds new `{class}_rf_reg_fraction` columns alongside the existing
`{class}_fraction` (NNLS) columns -- nothing from Section 4 is overwritten.

In [ ]:
tcga_rescaled = tcga_linear.values * scale_factor  # (n_samples, n_matched_genes)
rf_reg_raw = rf_reg.predict(np.arcsinh(tcga_rescaled))
rf_reg_fractions = softmax(rf_reg_raw)

rf_reg_df = pd.DataFrame(
    rf_reg_fractions,
    columns=[f'{c}_rf_reg_fraction' for c in classes],
    index=tcga_linear.index,
).reset_index().rename(columns={'index': 'sampleID'})

tcga_results_df = tcga_results_df.merge(rf_reg_df, on='sampleID')
tcga_results_df.to_csv('tcga_fusion_deconvolution_results.csv', index=False)
tcga_results_df.head()

## 5. Boxplot: estimated fused fraction by organ/tissue type

In [ ]:
MIN_SAMPLES_PER_ORGAN = 5

organ_counts = tcga_results_df['_primary_site'].value_counts()
kept_organs = organ_counts[organ_counts >= MIN_SAMPLES_PER_ORGAN].index
plot_df = tcga_results_df[tcga_results_df['_primary_site'].isin(kept_organs)]

organ_order = (
    plot_df.groupby('_primary_site')['fused_fraction']
    .median()
    .sort_values(ascending=False)
    .index
)

fig, ax = plt.subplots(figsize=(max(8, 0.4 * len(organ_order)), 6))
sns.boxplot(data=plot_df, x='_primary_site', y='fused_fraction', order=organ_order, ax=ax)
ax.set_xlabel('Organ / tissue type (primary site)')
ax.set_ylabel('Estimated fused-cell fraction (NNLS)')
ax.set_title('TCGA bulk RNA-seq: estimated fused fraction by organ')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

for i, organ in enumerate(organ_order):
    n = organ_counts[organ]
    ax.annotate(f'n={n}', (i, ax.get_ylim()[1]), ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('tcga_fused_fraction_by_organ.png', dpi=150)
plt.show()

## 5b. Compare NNLS vs. RF-Regression Estimates

TCGA has no ground-truth composition to validate against directly -- the held-out pseudo-bulk metrics
in Section 3c are the only calibrated performance numbers available for the RF regressor. What we
*can* check here is whether the two independent methods (NNLS against a fixed signature matrix vs. an
RF trained end-to-end on simulated mixtures) agree on real TCGA samples. Agreement isn't proof of
correctness -- both could share the same blind spots (e.g. the same cross-platform rescaling) -- but
disagreement would be a clear warning sign.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(tcga_results_df['fused_fraction'], tcga_results_df['fused_rf_reg_fraction'], alpha=0.3, s=10)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)

r, _ = pearsonr(tcga_results_df['fused_fraction'], tcga_results_df['fused_rf_reg_fraction'])
ax.text(
    0.05, 0.95, f'$r$ = {r:.3f}', transform=ax.transAxes, ha='left', va='top', fontsize=11,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'),
)
ax.set_xlabel('NNLS-estimated fused fraction')
ax.set_ylabel('RF-regression-estimated fused fraction')
ax.set_title('TCGA Samples: NNLS vs. RF Regression Agreement')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig('tcga_nnls_vs_rf_reg_agreement.png', dpi=150)
plt.show()

In [ ]:
organ_counts = tcga_results_df['_primary_site'].value_counts()
kept_organs = organ_counts[organ_counts >= MIN_SAMPLES_PER_ORGAN].index
plot_df_rf = tcga_results_df[tcga_results_df['_primary_site'].isin(kept_organs)]

organ_order_rf = (
    plot_df_rf.groupby('_primary_site')['fused_rf_reg_fraction']
    .median()
    .sort_values(ascending=False)
    .index
)

fig, ax = plt.subplots(figsize=(max(8, 0.4 * len(organ_order_rf)), 6))
sns.boxplot(data=plot_df_rf, x='_primary_site', y='fused_rf_reg_fraction', order=organ_order_rf, ax=ax)
ax.set_xlabel('Organ / tissue type (primary site)')
ax.set_ylabel('Estimated fused-cell fraction (RF regression)')
ax.set_title('TCGA bulk RNA-seq: estimated fused fraction by organ (RF regression)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

for i, organ in enumerate(organ_order_rf):
    n = organ_counts[organ]
    ax.annotate(f'n={n}', (i, ax.get_ylim()[1]), ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('tcga_fused_fraction_by_organ_rf_reg.png', dpi=150)
plt.show()

---
## Offline / HPC fallback (no `xenaPython`)

`xenaPython` isn't installed on the HPC. The block below is the equivalent full-download-and-subset
path: pulls the raw Xena bulk `.gz` matrix and phenotype file over plain HTTP, reads the (large) matrix
in row chunks so the full ~50k-gene x ~10.5k-sample matrix is never fully materialized, keeps only the
signature-gene rows, and writes intermediate/final outputs to
`/mnt/vstor/SOM_CCCC_JGS/shultesp/data`. Uses `tcga_RSEM_gene_tpm`'s Hugo-symbol sibling dataset
(`tcga_RSEM_Hugo_norm_count`) so rows are already gene symbols and no probeMap/Ensembl-ID join step is
needed.

Left commented out — uncomment to run on the HPC once `signature_genes` is available in that
environment (e.g. by also copying `fusion_signature_expression.h5ad` there, or hardcoding the gene list).

In [ ]:
# import os
# import requests
#
# DATA_DIR = '/mnt/vstor/SOM_CCCC_JGS/shultesp/data'
# os.makedirs(DATA_DIR, exist_ok=True)
#
# EXPR_URL = 'https://toil.xenahubs.net/download/tcga_RSEM_Hugo_norm_count.gz'
# PHENO_URL = 'https://toil.xenahubs.net/download/TcgaTargetGTEX_phenotype.txt.gz'
# EXPR_LOCAL = os.path.join(DATA_DIR, 'tcga_RSEM_Hugo_norm_count.gz')
# PHENO_LOCAL = os.path.join(DATA_DIR, 'TcgaTargetGTEX_phenotype.txt.gz')
#
# def download(url, path):
#     if os.path.exists(path):
#         return
#     with requests.get(url, stream=True) as r:
#         r.raise_for_status()
#         with open(path, 'wb') as f:
#             for chunk in r.iter_content(chunk_size=1 << 20):
#                 f.write(chunk)
#
# download(EXPR_URL, EXPR_LOCAL)
# download(PHENO_URL, PHENO_LOCAL)
#
# pheno_df = pd.read_csv(PHENO_LOCAL, sep='\t')
# pheno_df = pheno_df[pheno_df['_study'] == 'TCGA']
# pheno_df = pheno_df[pheno_df['_sample_type'].isin(SAMPLE_TYPES_KEPT)].reset_index(drop=True)
#
# CHUNKSIZE = 2000  # genes per chunk, not samples - this file is genes-as-rows
# matched_chunks = []
# reader = pd.read_csv(EXPR_LOCAL, sep='\t', index_col=0, chunksize=CHUNKSIZE)
# for chunk in reader:
#     hit = chunk.loc[chunk.index.isin(signature_genes)]
#     if not hit.empty:
#         matched_chunks.append(hit)
#
# tcga_log2norm = pd.concat(matched_chunks).T  # (n_samples, n_matched_genes)
# tcga_log2norm = tcga_log2norm.loc[tcga_log2norm.index.isin(pheno_df['sampleID'])]
# tcga_linear = 2 ** tcga_log2norm - 1
#
# tcga_linear.to_csv(os.path.join(DATA_DIR, 'tcga_signature_gene_expression_linear.csv'))
# pheno_df.to_csv(os.path.join(DATA_DIR, 'tcga_phenotype_filtered.csv'), index=False)
#
# # from here, join tcga_linear with pheno_df and reuse the NNLS + rescaling cell (Section 4) and the
# # RF regressor training/application cells (Sections 3b, 3c, 4b) above unchanged -- matched_genes
# # must be redefined here too (tcga_log2norm.columns, intersected with signature_genes) since the
# # Xena-streaming path that normally produces it is skipped entirely in this fallback